# GJR-GARCH Inverse Volatility Strategy with SMA50 + Correlation Filter
## Top 10 Cryptocurrencies Portfolio

---

### Strategy Components (Based on paul0805's approach + Rickdanoff's suggestion):

1. **Volatility Sizing**: Daily GARCH/EWMA calculation, inverse vol weighting
2. **Trend Filter**: SMA(50) - only invest when price > SMA
3. **Rebalancing**: WEEKLY (not daily) to reduce turnover
4. **Capital Allocation**: Only 50% invested, other 50% in cash earning 3%/year
5. **NEW: Correlation Filter**: Prefer assets with LOWER correlation to others (diversification)

### Why Correlation Matters:
- If all your cryptos move together, you're not truly diversified
- By weighting towards LESS correlated assets, we reduce portfolio volatility
- Combined with inverse volatility = double risk reduction

---

In [ ]:
# INSTALLATION - Uncomment if needed
# !pip install yfinance numpy pandas scipy matplotlib seaborn

In [ ]:
# IMPORTS
import yfinance as yf
import numpy as np
import pandas as pd
import warnings
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12

print("✅ Libraries loaded!")

In [ ]:
# ============================================================================
# CONFIGURATION (Updated based on paul0805's approach)
# ============================================================================

# Top 10 Cryptocurrencies
CRYPTO_TICKERS = [
    'BTC-USD', 'ETH-USD', 'BNB-USD', 'XRP-USD', 'ADA-USD',
    'SOL-USD', 'DOGE-USD', 'DOT-USD', 'MATIC-USD', 'AVAX-USD'
]

# Date range
START_DATE = '2020-01-01'
END_DATE = None

# Strategy parameters (paul0805's approach)
SMA_PERIOD = 50              # Trend filter
VOL_LOOKBACK = 14            # EWMA volatility lookback
CORR_LOOKBACK = 60           # Correlation lookback (60 days)
REBALANCE_FREQ = 'W'         # WEEKLY rebalancing (not daily)
CAPITAL_ALLOCATION = 0.50    # Only 50% of capital invested
CASH_RATE = 0.03             # 3% annual return on uninvested cash
MAX_LEVERAGE = 2.0           # Max leverage on the invested portion
TARGET_VOL = 0.15            # 15% target vol on invested portion

# Correlation filter parameters
USE_CORRELATION_FILTER = True  # Enable/disable correlation adjustment
CORR_WEIGHT = 0.5              # How much to weight correlation vs volatility (0-1)

# Backtesting
INITIAL_CAPITAL = 100_000
FEES = 0.001
TRAIN_RATIO = 0.60

print("📊 Configuration (paul0805 style + correlation):")
print(f"   Rebalancing: {REBALANCE_FREQ} (Weekly)")
print(f"   Capital Invested: {CAPITAL_ALLOCATION:.0%} (rest in cash @ {CASH_RATE:.0%}/yr)")
print(f"   Correlation Filter: {'ON' if USE_CORRELATION_FILTER else 'OFF'}")
print(f"   Trend Filter: SMA({SMA_PERIOD})")

In [ ]:
# ============================================================================
# DATA DOWNLOAD
# ============================================================================

print(f"Downloading data for {len(CRYPTO_TICKERS)} cryptocurrencies...")
print("="*70)

raw_data = yf.download(CRYPTO_TICKERS, start=START_DATE, end=END_DATE,
                       interval='1d', auto_adjust=True, progress=True)

if isinstance(raw_data.columns, pd.MultiIndex):
    close_prices = raw_data['Close']
else:
    close_prices = raw_data[['Close']]
    close_prices.columns = CRYPTO_TICKERS

# Data quality check
print("\n📊 Data Quality:")
valid_tickers = []
for ticker in CRYPTO_TICKERS:
    if ticker in close_prices.columns:
        col_data = close_prices[ticker].dropna()
        if len(col_data) >= 252:
            valid_tickers.append(ticker)
            print(f"  ✓ {ticker}: {len(col_data)} days")

close_prices = close_prices[valid_tickers].ffill(limit=5).dropna()
print(f"\n✅ Final: {len(valid_tickers)} assets, {len(close_prices)} days")

In [ ]:
# ============================================================================
# CALCULATE RETURNS, VOLATILITY, AND CORRELATIONS
# ============================================================================

# Returns
log_returns = np.log(close_prices / close_prices.shift(1)).dropna()
simple_returns = close_prices.pct_change().dropna()

# Daily EWMA Volatility
volatility_df = log_returns.ewm(span=VOL_LOOKBACK, min_periods=VOL_LOOKBACK).std()

# Rolling Correlation Matrix (average correlation of each asset to others)
def calculate_avg_correlation(returns_df, lookback):
    """
    Calculate the average correlation of each asset to all OTHER assets.
    Lower correlation = better diversification = higher weight.
    """
    avg_corr = pd.DataFrame(index=returns_df.index, columns=returns_df.columns, dtype=float)
    
    for i in range(lookback, len(returns_df)):
        window = returns_df.iloc[i-lookback:i]
        corr_matrix = window.corr()
        
        # For each asset, calculate average correlation to OTHER assets
        for col in returns_df.columns:
            other_corrs = corr_matrix[col].drop(col)  # Exclude self-correlation
            avg_corr.loc[returns_df.index[i], col] = other_corrs.mean()
    
    return avg_corr

print("Calculating rolling correlations (this may take a moment)...")
avg_correlation = calculate_avg_correlation(log_returns, CORR_LOOKBACK)

print("\nAverage Correlation to Other Assets:")
for ticker in valid_tickers:
    mean_corr = avg_correlation[ticker].dropna().mean()
    print(f"  {ticker}: {mean_corr:.2f}")

In [ ]:
# ============================================================================
# SMA50 TREND FILTER
# ============================================================================

sma_values = close_prices.rolling(window=SMA_PERIOD, min_periods=SMA_PERIOD).mean()
trend_filter = close_prices > sma_values

print(f"SMA({SMA_PERIOD}) Trend Filter - % Time in Uptrend:")
for ticker in valid_tickers:
    uptrend_pct = trend_filter[ticker].dropna().mean() * 100
    print(f"  {ticker}: {uptrend_pct:.1f}%")

In [ ]:
# ============================================================================
# INVERSE VOLATILITY + CORRELATION WEIGHTS
# ============================================================================

def calculate_combined_weights(vol_df, corr_df, trend_df, 
                                use_corr=True, corr_weight=0.5,
                                max_lev=2.0, target_vol=0.15):
    """
    Calculate weights using:
    1. Inverse Volatility (lower vol = higher weight)
    2. Inverse Correlation (lower corr to others = higher weight)
    3. Trend Filter (must be in uptrend)
    
    Combined score = (1-corr_weight) * inv_vol_score + corr_weight * inv_corr_score
    """
    common_idx = vol_df.index.intersection(trend_df.index)
    if use_corr:
        common_idx = common_idx.intersection(corr_df.index)
    
    vol = vol_df.loc[common_idx].copy()
    trend = trend_df.loc[common_idx].copy()
    
    if use_corr:
        corr = corr_df.loc[common_idx].copy()
    
    # Apply trend filter
    filtered_vol = vol.copy()
    filtered_vol[~trend] = np.inf
    
    # Inverse volatility score (normalized)
    inv_vol = 1.0 / filtered_vol
    inv_vol = inv_vol.replace([np.inf, -np.inf], 0)
    inv_vol_norm = inv_vol.div(inv_vol.sum(axis=1), axis=0).fillna(0)
    
    if use_corr:
        # Inverse correlation score (lower correlation = better)
        # Transform: correlation is typically 0-1, we want inverse
        filtered_corr = corr.copy()
        filtered_corr[~trend] = 1.0  # Max correlation for filtered assets
        
        # Inverse correlation: 1 - corr (so lower corr = higher score)
        inv_corr = 1.0 - filtered_corr
        inv_corr = inv_corr.clip(lower=0.01)  # Avoid zeros
        inv_corr[~trend] = 0  # Zero weight for filtered
        inv_corr_norm = inv_corr.div(inv_corr.sum(axis=1), axis=0).fillna(0)
        
        # Combined score
        combined_score = (1 - corr_weight) * inv_vol_norm + corr_weight * inv_corr_norm
    else:
        combined_score = inv_vol_norm
    
    # Normalize to sum to 1
    row_sums = combined_score.sum(axis=1)
    weights = combined_score.div(row_sums, axis=0).fillna(0)
    
    # Volatility scaling
    daily_target = target_vol / np.sqrt(365)
    portfolio_vol = (weights * vol).sum(axis=1)
    vol_scale = daily_target / portfolio_vol.replace(0, np.inf)
    vol_scale = vol_scale.clip(upper=max_lev).replace([np.inf, -np.inf], 0)
    
    scaled_weights = weights.multiply(vol_scale, axis=0)
    
    # Cap leverage
    total_exp = scaled_weights.sum(axis=1)
    lev_scale = (max_lev / total_exp).clip(upper=1.0)
    final_weights = scaled_weights.multiply(lev_scale, axis=0)
    
    return final_weights

print("Calculating combined weights (vol + correlation)...")

In [ ]:
# ============================================================================
# CALCULATE WEIGHTS WITH WEEKLY REBALANCING
# ============================================================================

# Calculate daily weights first
daily_weights = calculate_combined_weights(
    volatility_df, avg_correlation, trend_filter,
    use_corr=USE_CORRELATION_FILTER, corr_weight=CORR_WEIGHT,
    max_lev=MAX_LEVERAGE, target_vol=TARGET_VOL
)

# Apply 1-day lag to prevent lookahead bias
daily_weights = daily_weights.shift(1).dropna()

# Convert to WEEKLY rebalancing (hold weights constant within each week)
# Resample to weekly, taking the first day of each week's weights
weekly_weights = daily_weights.resample('W').first()

# Forward fill to daily frequency (hold weights constant throughout the week)
portfolio_weights = weekly_weights.reindex(daily_weights.index, method='ffill')

# Apply capital allocation (only 50% invested)
portfolio_weights = portfolio_weights * CAPITAL_ALLOCATION

print(f"\n✅ Weekly rebalancing applied")
print(f"   Capital allocated: {CAPITAL_ALLOCATION:.0%}")
print(f"   Average Exposure: {portfolio_weights.sum(axis=1).mean():.1%}")
print(f"   Rebalance events: {len(weekly_weights)}")

In [ ]:
# ============================================================================
# PREPARE DATA FOR BACKTESTING
# ============================================================================

common_index = close_prices.index.intersection(portfolio_weights.index).intersection(simple_returns.index)
prices_aligned = close_prices.loc[common_index]
weights_aligned = portfolio_weights.loc[common_index]
returns_aligned = simple_returns.loc[common_index]

# Train/Val split
split_idx = int(len(common_index) * TRAIN_RATIO)
train_end_date = common_index[split_idx]

train_weights = weights_aligned.iloc[:split_idx]
train_returns = returns_aligned.iloc[:split_idx]
val_weights = weights_aligned.iloc[split_idx:]
val_returns = returns_aligned.iloc[split_idx:]

print(f"Training:   {common_index[0].date()} → {train_end_date.date()} ({split_idx} days)")
print(f"Validation: {common_index[split_idx].date()} → {common_index[-1].date()} ({len(common_index)-split_idx} days)")

In [ ]:
# ============================================================================
# BACKTESTING FUNCTION (with cash return)
# ============================================================================

def backtest_portfolio_with_cash(weights, returns, init_cash=100000, fees=0.001, 
                                  cash_rate=0.03):
    """
    Backtest portfolio with:
    - Transaction costs
    - Cash earning interest on uninvested portion
    """
    # Crypto returns (weighted)
    crypto_returns = (weights * returns).sum(axis=1)
    
    # Cash portion returns (uninvested capital earns cash_rate)
    cash_weight = 1.0 - weights.sum(axis=1)
    daily_cash_rate = cash_rate / 365
    cash_returns = cash_weight * daily_cash_rate
    
    # Total portfolio returns
    portfolio_returns = crypto_returns + cash_returns
    
    # Transaction costs (only on rebalancing days)
    turnover = weights.diff().abs().sum(axis=1).fillna(0)
    net_returns = portfolio_returns - (turnover * fees)
    
    # Equity curve
    equity_curve = init_cash * (1 + net_returns).cumprod()
    
    # Metrics
    total_return = equity_curve.iloc[-1] / init_cash - 1
    n_years = len(net_returns) / 365
    ann_return = (1 + total_return) ** (1/n_years) - 1
    daily_vol = net_returns.std()
    ann_vol = daily_vol * np.sqrt(365)
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0
    
    rolling_max = equity_curve.expanding().max()
    drawdown = (equity_curve - rolling_max) / rolling_max
    max_drawdown = drawdown.min()
    
    downside_vol = net_returns[net_returns < 0].std() * np.sqrt(365)
    sortino = ann_return / downside_vol if downside_vol > 0 else 0
    calmar = ann_return / abs(max_drawdown) if max_drawdown != 0 else 0
    
    # Calculate actual rebalances (significant weight changes)
    rebalances = (turnover > 0.01).sum()
    
    return {
        'equity_curve': equity_curve,
        'returns': net_returns,
        'drawdown': drawdown,
        'total_return': total_return,
        'ann_return': ann_return,
        'ann_vol': ann_vol,
        'sharpe': sharpe,
        'sortino': sortino,
        'calmar': calmar,
        'max_drawdown': max_drawdown,
        'avg_exposure': weights.sum(axis=1).mean(),
        'avg_turnover': turnover.mean() * 365,
        'rebalances': rebalances
    }

In [ ]:
# ============================================================================
# RUN BACKTESTS
# ============================================================================

# Strategy backtests (with correlation filter)
train_results = backtest_portfolio_with_cash(train_weights, train_returns, 
                                              INITIAL_CAPITAL, FEES, CASH_RATE)
val_results = backtest_portfolio_with_cash(val_weights, val_returns, 
                                            INITIAL_CAPITAL, FEES, CASH_RATE)
full_results = backtest_portfolio_with_cash(weights_aligned, returns_aligned, 
                                             INITIAL_CAPITAL, FEES, CASH_RATE)

# BENCHMARK 1: Equal-weight Buy & Hold
n_assets = len(valid_tickers)
benchmark_weights = pd.DataFrame(1.0/n_assets, index=weights_aligned.index, columns=valid_tickers)
benchmark_full = backtest_portfolio_with_cash(benchmark_weights, returns_aligned, 
                                               INITIAL_CAPITAL, FEES, 0)

# BENCHMARK 2: Strategy WITHOUT correlation filter (for comparison)
weights_no_corr = calculate_combined_weights(
    volatility_df, avg_correlation, trend_filter,
    use_corr=False,  # Disable correlation
    max_lev=MAX_LEVERAGE, target_vol=TARGET_VOL
)
weights_no_corr = weights_no_corr.shift(1).dropna()
weekly_no_corr = weights_no_corr.resample('W').first()
weights_no_corr = weekly_no_corr.reindex(weights_no_corr.index, method='ffill')
weights_no_corr = weights_no_corr * CAPITAL_ALLOCATION
weights_no_corr = weights_no_corr.loc[weights_aligned.index]

no_corr_results = backtest_portfolio_with_cash(weights_no_corr, returns_aligned, 
                                                INITIAL_CAPITAL, FEES, CASH_RATE)

print("✅ Backtests completed!")
print(f"   Strategy rebalances: {full_results['rebalances']}")

In [ ]:
# ============================================================================
# RESULTS COMPARISON
# ============================================================================

print("\n" + "="*100)
print(" STRATEGY COMPARISON: With vs Without Correlation Filter")
print("="*100)
print(f"\n{'Metric':<25} | {'With Corr Filter':>18} | {'Without Corr':>18} | {'Benchmark B&H':>18}")
print("-"*100)
print(f"{'Total Return':<25} | {full_results['total_return']*100:>17.1f}% | {no_corr_results['total_return']*100:>17.1f}% | {benchmark_full['total_return']*100:>17.1f}%")
print(f"{'CAGR':<25} | {full_results['ann_return']*100:>17.1f}% | {no_corr_results['ann_return']*100:>17.1f}% | {benchmark_full['ann_return']*100:>17.1f}%")
print(f"{'Sharpe Ratio':<25} | {full_results['sharpe']:>18.2f} | {no_corr_results['sharpe']:>18.2f} | {benchmark_full['sharpe']:>18.2f}")
print(f"{'Sortino Ratio':<25} | {full_results['sortino']:>18.2f} | {no_corr_results['sortino']:>18.2f} | {benchmark_full['sortino']:>18.2f}")
print(f"{'Max Drawdown':<25} | {full_results['max_drawdown']*100:>17.1f}% | {no_corr_results['max_drawdown']*100:>17.1f}% | {benchmark_full['max_drawdown']*100:>17.1f}%")
print(f"{'Annualized Vol':<25} | {full_results['ann_vol']*100:>17.1f}% | {no_corr_results['ann_vol']*100:>17.1f}% | {benchmark_full['ann_vol']*100:>17.1f}%")
print(f"{'Average Exposure':<25} | {full_results['avg_exposure']*100:>17.1f}% | {no_corr_results['avg_exposure']*100:>17.1f}% | {benchmark_full['avg_exposure']*100:>17.1f}%")
print(f"{'Calmar Ratio':<25} | {full_results['calmar']:>18.2f} | {no_corr_results['calmar']:>18.2f} | {benchmark_full['calmar']:>18.2f}")
print("="*100)

# Impact of correlation filter
sharpe_improvement = (full_results['sharpe'] - no_corr_results['sharpe']) / no_corr_results['sharpe'] * 100
dd_improvement = (full_results['max_drawdown'] - no_corr_results['max_drawdown']) / abs(no_corr_results['max_drawdown']) * 100

print(f"\n📊 CORRELATION FILTER IMPACT:")
print(f"   Sharpe Ratio: {sharpe_improvement:+.1f}% {'improvement' if sharpe_improvement > 0 else 'change'}")
print(f"   Max Drawdown: {dd_improvement:+.1f}% {'improvement (less negative)' if dd_improvement > 0 else 'change'}")

In [ ]:
# ============================================================================
# COMPREHENSIVE DASHBOARD
# ============================================================================

fig = plt.figure(figsize=(20, 24))
gs = fig.add_gridspec(6, 3, hspace=0.35, wspace=0.25)

# 1. CUMULATIVE RETURNS - All strategies
ax1 = fig.add_subplot(gs[0, :2])
strat_cum = (1 + full_results['returns']).cumprod() - 1
no_corr_cum = (1 + no_corr_results['returns']).cumprod() - 1
bench_cum = (1 + benchmark_full['returns']).cumprod() - 1

ax1.fill_between(strat_cum.index, 0, strat_cum.values * 100, alpha=0.3, color='#2ecc71')
ax1.plot(strat_cum.index, strat_cum.values * 100, color='#27ae60', linewidth=2.5, label='With Corr Filter')
ax1.plot(no_corr_cum.index, no_corr_cum.values * 100, color='#3498db', linewidth=2, alpha=0.8, label='Without Corr Filter')
ax1.plot(bench_cum.index, bench_cum.values * 100, color='#e74c3c', linewidth=2, alpha=0.6, label='Benchmark B&H')
ax1.axvline(x=train_end_date, color='black', linestyle='--', alpha=0.5, label='Train/Val Split')
ax1.set_ylabel('Return (%)')
ax1.set_title('Cumulative Returns Comparison', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# 2. KEY METRICS RECEIPT
ax_receipt = fig.add_subplot(gs[0, 2])
ax_receipt.axis('off')
receipt_text = f"""
┌─────────────────────────────┐
│   KEY METRICS (w/ Corr)    │
│─────────────────────────────│
│ Total Return:  {full_results['total_return']*100:>10.1f}% │
│ CAGR:          {full_results['ann_return']*100:>10.1f}% │
│ Sharpe:        {full_results['sharpe']:>10.2f}  │
│ Sortino:       {full_results['sortino']:>10.2f}  │
│ Max DD:        {full_results['max_drawdown']*100:>10.1f}% │
│ Avg Exposure:  {full_results['avg_exposure']*100:>10.1f}% │
│ Ann. Vol:      {full_results['ann_vol']*100:>10.1f}% │
│ Calmar:        {full_results['calmar']:>10.2f}  │
│ Rebalances:    {full_results['rebalances']:>10}  │
└─────────────────────────────┘
"""
ax_receipt.text(0.5, 0.5, receipt_text, transform=ax_receipt.transAxes,
                fontsize=11, fontfamily='monospace', 
                verticalalignment='center', horizontalalignment='center',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='#fffef0', 
                         edgecolor='#333333', linewidth=2))

# 3. DRAWDOWN COMPARISON
ax2 = fig.add_subplot(gs[1, :2])
ax2.fill_between(full_results['drawdown'].index, full_results['drawdown'].values * 100, 0,
                 alpha=0.6, color='#27ae60', label='With Corr Filter')
ax2.fill_between(benchmark_full['drawdown'].index, benchmark_full['drawdown'].values * 100, 0,
                 alpha=0.3, color='#e74c3c', label='Benchmark B&H')
ax2.set_ylabel('Drawdown (%)')
ax2.set_title('Drawdown Comparison', fontsize=14, fontweight='bold')
ax2.legend(loc='lower left')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(min(benchmark_full['drawdown'].min() * 100 - 5, -85), 5)

# 4. CORRELATION HEATMAP
ax3 = fig.add_subplot(gs[1, 2])
recent_corr = log_returns.tail(CORR_LOOKBACK).corr()
mask = np.triu(np.ones_like(recent_corr, dtype=bool))
sns.heatmap(recent_corr, mask=mask, annot=True, fmt='.2f', center=0.5,
            cmap='RdYlGn_r', linewidths=0.5, ax=ax3,
            cbar_kws={'label': 'Correlation', 'shrink': 0.8},
            annot_kws={'size': 7})
ax3.set_title(f'Asset Correlations (Last {CORR_LOOKBACK}d)', fontsize=14, fontweight='bold')

# 5. EQUITY CURVES ($100 Investment)
ax4 = fig.add_subplot(gs[2, :2])
strat_equity = 100 * (1 + full_results['returns']).cumprod()
no_corr_equity = 100 * (1 + no_corr_results['returns']).cumprod()
bench_equity = 100 * (1 + benchmark_full['returns']).cumprod()

ax4.plot(strat_equity.index, strat_equity.values, color='#27ae60', linewidth=2.5,
         label=f'With Corr (${strat_equity.iloc[-1]:.0f})')
ax4.plot(no_corr_equity.index, no_corr_equity.values, color='#3498db', linewidth=2,
         label=f'Without Corr (${no_corr_equity.iloc[-1]:.0f})')
ax4.plot(bench_equity.index, bench_equity.values, color='#e74c3c', linewidth=2, alpha=0.7,
         label=f'Benchmark (${bench_equity.iloc[-1]:.0f})')
ax4.axvline(x=train_end_date, color='black', linestyle='--', alpha=0.5)
ax4.set_ylabel('Portfolio Value ($)')
ax4.set_title('$100 Investment Growth', fontsize=14, fontweight='bold')
ax4.legend(loc='upper left')
ax4.set_yscale('log')
ax4.grid(True, alpha=0.3)
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# 6. YEARLY RETURNS
ax5 = fig.add_subplot(gs[2, 2])
annual_rets = full_results['returns'].resample('Y').apply(lambda x: (1+x).prod()-1)
years = [d.year for d in annual_rets.index]
colors = ['#27ae60' if r >= 0 else '#e74c3c' for r in annual_rets.values]
bars = ax5.bar(range(len(years)), annual_rets.values * 100, color=colors, edgecolor='black')
ax5.axhline(y=full_results['ann_return']*100, color='blue', linestyle='--', 
            label=f'CAGR: {full_results["ann_return"]*100:.1f}%')
ax5.set_ylabel('Return (%)')
ax5.set_title('Yearly Results', fontsize=14, fontweight='bold')
ax5.set_xticks(range(len(years)))
ax5.set_xticklabels(years, rotation=45)
ax5.legend()
for bar, val in zip(bars, annual_rets.values * 100):
    ypos = val + 2 if val >= 0 else val - 8
    ax5.text(bar.get_x() + bar.get_width()/2, ypos, f'{val:.0f}%', ha='center', fontsize=9, fontweight='bold')

# 7. EXPOSURE OVER TIME
ax6 = fig.add_subplot(gs[3, :2])
exposure = weights_aligned.sum(axis=1) * 100
ax6.fill_between(exposure.index, 0, exposure.values, alpha=0.6, color='#3498db')
ax6.axhline(y=50, color='orange', linestyle='--', label='50% Max (Capital Alloc)')
ax6.axhline(y=exposure.mean(), color='green', linestyle='-', label=f'Average: {exposure.mean():.1f}%')
ax6.set_ylabel('Exposure (%)')
ax6.set_title('Dynamic Exposure (50% Capital Limit Applied)', fontsize=14, fontweight='bold')
ax6.legend(loc='upper right')
ax6.set_ylim(0, 60)

# 8. ASSET WEIGHTS (Average)
ax7 = fig.add_subplot(gs[3, 2])
avg_weights = weights_aligned.mean().sort_values(ascending=True)
if avg_weights.sum() > 0:
    avg_weights_pct = avg_weights / avg_weights.sum() * 100
else:
    avg_weights_pct = avg_weights * 0
colors_alloc = plt.cm.Greens(np.linspace(0.3, 0.9, len(avg_weights)))
bars = ax7.barh(avg_weights.index, avg_weights_pct.values, color=colors_alloc, edgecolor='black')
ax7.set_xlabel('Avg Allocation (%)')
ax7.set_title('Asset Allocation (Avg)', fontsize=14, fontweight='bold')
for bar, val in zip(bars, avg_weights_pct.values):
    if val > 0:
        ax7.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=9)

# 9. RETURNS DISTRIBUTION
ax8 = fig.add_subplot(gs[4, 0])
returns_pct = full_results['returns'] * 100
ax8.hist(returns_pct, bins=50, density=True, alpha=0.7, color='#27ae60', edgecolor='black')
mu, std = returns_pct.mean(), returns_pct.std()
x_dist = np.linspace(returns_pct.min(), returns_pct.max(), 100)
ax8.plot(x_dist, stats.norm.pdf(x_dist, mu, std), 'r--', linewidth=2, label='Normal')
ax8.axvline(x=mu, color='green', linestyle='-', label=f'Mean: {mu:.2f}%')
ax8.set_xlabel('Daily Return (%)')
ax8.set_ylabel('Density')
ax8.set_title('Daily Returns Distribution', fontsize=14, fontweight='bold')
ax8.legend(fontsize=9)

# 10. ROLLING SHARPE
ax9 = fig.add_subplot(gs[4, 1])
rolling_sharpe = full_results['returns'].rolling(252).apply(
    lambda x: (x.mean() * 365) / (x.std() * np.sqrt(365)) if x.std() > 0 else 0)
ax9.plot(rolling_sharpe.index, rolling_sharpe.values, color='#9b59b6', linewidth=1.5)
ax9.axhline(y=full_results['sharpe'], color='#9b59b6', linestyle='--', label=f'Overall: {full_results["sharpe"]:.2f}')
ax9.axhline(y=2.0, color='green', linestyle=':', alpha=0.5, label='SR = 2.0 (Target)')
ax9.axhline(y=0, color='red', linestyle=':', alpha=0.5)
ax9.set_ylabel('Sharpe Ratio')
ax9.set_title('Rolling Sharpe (252d)', fontsize=14, fontweight='bold')
ax9.legend(fontsize=9)

# 11. ROLLING VOLATILITY
ax10 = fig.add_subplot(gs[4, 2])
rolling_vol = full_results['returns'].rolling(252).std() * np.sqrt(365) * 100
ax10.plot(rolling_vol.index, rolling_vol.values, color='#e67e22', linewidth=1.5)
ax10.axhline(y=full_results['ann_vol']*100, color='#e67e22', linestyle='--', label=f'Overall: {full_results["ann_vol"]*100:.1f}%')
ax10.axhline(y=TARGET_VOL*100, color='green', linestyle=':', alpha=0.5, label=f'Target: {TARGET_VOL*100:.0f}%')
ax10.set_ylabel('Ann. Volatility (%)')
ax10.set_title('Rolling Volatility (252d)', fontsize=14, fontweight='bold')
ax10.legend(fontsize=9)

# 12. STRATEGY SUMMARY TABLE
ax11 = fig.add_subplot(gs[5, :])
ax11.axis('off')

summary_text = f"""
═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
                              STRATEGY SUMMARY: Inverse Vol + Correlation Filter + SMA50 Trend
═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

 STRATEGY COMPONENTS:
 ────────────────────
 1. SMA(50) Trend Filter    → Only invest when price > 50-day moving average
 2. Inverse Volatility      → Allocate MORE to calm assets, LESS to volatile ones
 3. Correlation Filter      → Prefer assets with LOWER correlation to others (diversification)
 4. Weekly Rebalancing      → Reduces turnover and transaction costs
 5. 50% Capital Allocation  → Only half invested, rest in cash earning 3%/year

 ┌────────────────────────┬─────────────────────┬─────────────────────┬─────────────────────┐
 │ Metric                 │ WITH Corr Filter    │ WITHOUT Corr Filter │ Benchmark (B&H)     │
 ├────────────────────────┼─────────────────────┼─────────────────────┼─────────────────────┤
 │ Total Return           │ {full_results['total_return']*100:>16.1f}%   │ {no_corr_results['total_return']*100:>16.1f}%   │ {benchmark_full['total_return']*100:>16.1f}%   │
 │ Sharpe Ratio           │ {full_results['sharpe']:>17.2f}    │ {no_corr_results['sharpe']:>17.2f}    │ {benchmark_full['sharpe']:>17.2f}    │
 │ Max Drawdown           │ {full_results['max_drawdown']*100:>16.1f}%   │ {no_corr_results['max_drawdown']*100:>16.1f}%   │ {benchmark_full['max_drawdown']*100:>16.1f}%   │
 │ Average Exposure       │ {full_results['avg_exposure']*100:>16.1f}%   │ {no_corr_results['avg_exposure']*100:>16.1f}%   │ {benchmark_full['avg_exposure']*100:>16.1f}%   │
 └────────────────────────┴─────────────────────┴─────────────────────┴─────────────────────┘

 KEY TAKEAWAY: The correlation filter helps diversify risk by preferring assets that don't move together.
               Combined with inverse volatility and trend filtering, this creates a robust risk-managed portfolio.

═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
"""
ax11.text(0.5, 0.5, summary_text, transform=ax11.transAxes, fontsize=9, fontfamily='monospace',
          verticalalignment='center', horizontalalignment='center',
          bbox=dict(boxstyle='round,pad=0.3', facecolor='#f8f9fa', edgecolor='#333333', linewidth=1))

plt.suptitle('GJR-GARCH Inverse Vol + Correlation Filter - Performance Dashboard', 
             fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('performance_dashboard_with_corr.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("\n📊 Dashboard saved!")

In [ ]:
# ============================================================================
# RECEIPT PRINTOUT
# ============================================================================

fig_r, ax_r = plt.subplots(figsize=(4, 6))
ax_r.axis('off')

receipt = f"""
╔═══════════════════════════════╗
║                               ║
║   STRATEGY PERFORMANCE        ║
║   ════════════════════        ║
║                               ║
║  Total Return:   {full_results['total_return']*100:>8.1f}%   ║
║  CAGR:           {full_results['ann_return']*100:>8.1f}%   ║
║  Sharpe:         {full_results['sharpe']:>8.2f}    ║
║  Sortino:        {full_results['sortino']:>8.2f}    ║
║  Max DD:         {full_results['max_drawdown']*100:>8.1f}%   ║
║  Avg Exposure:   {full_results['avg_exposure']*100:>8.1f}%   ║
║  Ann. Vol:       {full_results['ann_vol']*100:>8.1f}%   ║
║  Calmar:         {full_results['calmar']:>8.2f}    ║
║  Rebalances:     {full_results['rebalances']:>8}    ║
║                               ║
║  ─────────────────────────    ║
║  Weekly Rebalancing           ║
║  50% Capital Allocation       ║
║  Correlation Filter: ON       ║
║                               ║
║  {prices_aligned.index[0].strftime('%Y-%m-%d')} → {prices_aligned.index[-1].strftime('%Y-%m-%d')}     ║
║                               ║
╚═══════════════════════════════╝
"""

ax_r.text(0.5, 0.5, receipt, transform=ax_r.transAxes, fontsize=12, fontfamily='monospace',
          verticalalignment='center', horizontalalignment='center',
          bbox=dict(boxstyle='round,pad=0.5', facecolor='#fffef0', edgecolor='#2c3e50', linewidth=3))

plt.tight_layout()
plt.savefig('metrics_receipt_with_corr.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("📄 Receipt saved!")

In [ ]:
# ============================================================================
# FINAL LAYMAN'S SUMMARY
# ============================================================================

print("\n" + "="*80)
print(" FINAL SUMMARY - EXPLAINED SIMPLY")
print("="*80)

print(f"""
🎯 WHAT THIS STRATEGY DOES:
───────────────────────────
1. Only buys crypto when it's TRENDING UP (price > 50-day average)
2. Buys MORE of CALM cryptos, LESS of VOLATILE ones
3. Prefers cryptos that DON'T move together (diversification)
4. Only invests HALF your money (rest stays safe in cash)
5. Rebalances WEEKLY (not daily) to save on fees

📊 THE RESULTS ({prices_aligned.index[0].date()} → {prices_aligned.index[-1].date()}):
───────────────────────────────────────────────────────────────────────────────

                    WITH CORR       WITHOUT CORR      BUY & HOLD
                    FILTER          FILTER            (BENCHMARK)
                    ─────────       ────────────      ──────────
Total Return:       {full_results['total_return']*100:>8.1f}%        {no_corr_results['total_return']*100:>8.1f}%         {benchmark_full['total_return']*100:>8.1f}%
Sharpe Ratio:       {full_results['sharpe']:>8.2f}          {no_corr_results['sharpe']:>8.2f}           {benchmark_full['sharpe']:>8.2f}
Max Drawdown:       {full_results['max_drawdown']*100:>8.1f}%        {no_corr_results['max_drawdown']*100:>8.1f}%         {benchmark_full['max_drawdown']*100:>8.1f}%
Avg Exposure:       {full_results['avg_exposure']*100:>8.1f}%        {no_corr_results['avg_exposure']*100:>8.1f}%         {benchmark_full['avg_exposure']*100:>8.1f}%

💡 WHAT THE CORRELATION FILTER ADDS:
────────────────────────────────────
• If BTC crashes, ETH probably crashes too (high correlation)
• By preferring LESS correlated assets, we don't put all eggs in one basket
• Result: {sharpe_improvement:+.1f}% change in Sharpe Ratio
• Result: {dd_improvement:+.1f}% change in Max Drawdown

🔑 THE KEY INSIGHT:
───────────────────
• Paul0805's approach: Simple rules, no optimization, just logic
• Rick's suggestion: Add correlation = better diversification
• 50% in cash = you sleep better at night
• Weekly rebalancing = lower costs than daily

⚠️ IMPORTANT CAVEATS:
─────────────────────
• Past performance ≠ future results
• This is backtested data, not live trading
• Crypto markets are highly speculative
• The parameters were NOT optimized (paul0805's point)
""")

print("="*80)
print("\n✅ Analysis complete!")